# RHESSI (Lin et al. 2002) — Implementation / 구현

**Paper**: Lin, R. P., Dennis, B. R., Hurford, G. J., et al., *The Reuven Ramaty High-Energy Solar Spectroscopic Imager (RHESSI)*, Solar Physics 210, 3-32, 2002. DOI: 10.1023/A:1022428818870.

This notebook implements three key concepts from the RHESSI paper.
이 노트북은 RHESSI 논문의 세 가지 핵심 개념을 구현한다.

1. **RMC modulation profile** for an off-axis point source on a spinning spacecraft.
   회전하는 우주선에서 축 외 점원에 대한 RMC 변조 프로파일.
2. **Back-projection imaging** from a synthetic two-source flare using 9 RMC pitches.
   9개 RMC pitch를 사용한 합성 이중 source 플레어의 back-projection 영상화.
3. **Hard X-ray bremsstrahlung spectrum** with thin-target / thick-target conversion and a thermal+nonthermal composite.
   Thin-target / thick-target 변환과 thermal + nonthermal 합성 hard X-ray bremsstrahlung 스펙트럼.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

rng = np.random.default_rng(42)

## Part 1: RMC Modulation Profile / RMC 변조 프로파일

We simulate the time-modulated count rate seen by a single RMC as the spacecraft spins. For a point source at off-axis angle $\theta$ and position angle $\phi_0$, the transmitted count rate as a function of rotation phase $\phi$ is approximately

$$M(\phi) = \tfrac{C_0}{2}\Bigl[1 + \cos\bigl(\tfrac{2\pi L}{p}\sin\theta \cdot \cos(\phi-\phi_0)\bigr)\Bigr]$$

where $L = 1.55$ m is the grid separation and $p$ is the grid pitch.

회전하는 우주선에서 단일 RMC가 보는 시간 변조 계수율을 시뮬레이션한다. 축 외 각 $\theta$, 위치각 $\phi_0$인 점원에 대해 회전 위상 $\phi$의 함수로 위 식과 같다.

In [ ]:
def rmc_modulation(phi, theta, phi0, pitch, L=1.55, C0=1.0):
    """Idealised RMC modulation profile for a point source.

    Args:
      phi: array of rotation phases [rad].
      theta: source off-axis angle [rad].
      phi0: source position angle [rad].
      pitch: grid pitch [m].
      L: grid separation [m] (default 1.55 m for RHESSI).
      C0: unmodulated count rate amplitude.

    Returns:
      Modulated count rate, same shape as phi.
    """
    arg = (2.0 * np.pi * L / pitch) * np.sin(theta) * np.cos(phi - phi0)
    return 0.5 * C0 * (1.0 + np.cos(arg))


ARCSEC = np.pi / (180.0 * 3600.0)  # radians per arcsec

# Two RHESSI grids: finest (34 um) and coarsest (2.75 mm)
phi = np.linspace(0.0, 2.0 * np.pi, 2000)
theta = 30.0 * ARCSEC  # source 30 arcsec off-axis
phi0 = np.deg2rad(45.0)

M_fine = rmc_modulation(phi, theta, phi0, pitch=34e-6)
M_coarse = rmc_modulation(phi, theta, phi0, pitch=2.75e-3)

fig, ax = plt.subplots()
ax.plot(np.rad2deg(phi), M_fine, label='Grid 1 (p = 34 um)', lw=1.5)
ax.plot(np.rad2deg(phi), M_coarse, label='Grid 9 (p = 2.75 mm)', lw=1.5)
ax.set_xlabel('Rotation phase [deg]')
ax.set_ylabel('Transmitted count rate (normalised)')
ax.set_title('RMC modulation profile, source at 30 arcsec off-axis')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The fine grid (34 µm pitch) modulates rapidly — its angular resolution $p/(2L) \approx 2.3$ arcsec means a 30-arcsec source spans ~13 fringe periods per radian of rotation. The coarse grid produces only one or two fringes — appropriate for resolving 3-arcmin scale features but blind to fine structure.

미세 grid (34 µm)는 분해능 $p/(2L)\approx 2.3$ arcsec, 회전당 ~13 fringe. 거친 grid는 fringe가 1-2개로 3 arcmin 스케일에 적합하지만 미세 구조에는 둔감하다.

In [ ]:
# Verify p/(2L) angular resolution numerically
for p in [34e-6, 2.75e-3]:
    res_arcsec = p / (2.0 * 1.55) / ARCSEC
    print(f'pitch = {p*1e6:.0f} um  ->  Delta theta = {res_arcsec:.2f} arcsec')

## Part 2: Back-Projection Imaging / Back-projection 영상 재구성

We simulate a flare consisting of two compact footpoints, generate the modulated count rates seen by 9 RHESSI RMCs, and reconstruct the image by *back-projection* — the simplest of the standard RHESSI algorithms.

Back-projection sums, for each candidate sky pixel, the modulation amplitude expected if a source were at that pixel. The result is the *dirty image* (the sky convolved with the instrument's point spread function); CLEAN/MEM/Pixon iteratively deconvolve this dirty image.

두 개의 compact footpoint로 이루어진 플레어를 시뮬레이션해 9개 RMC의 변조 신호를 생성하고, *back-projection*으로 영상을 재구성한다. 각 후보 sky 픽셀에 대해, 그 위치에 source가 있을 때의 변조 진폭을 합산한다. 결과는 *dirty image* (실제 sky를 PSF로 convolve한 것)이며, CLEAN/MEM/Pixon이 이를 deconvolve.

In [ ]:
# Define 9 RHESSI grid pitches in geometric progression of sqrt(3)
pitches = 34e-6 * np.sqrt(3) ** np.arange(9)  # 34 um, 59 um, ..., 2.75 mm
L = 1.55

# True source: two footpoints in arcsec coordinates
src_x_as = np.array([-15.0, +15.0])
src_y_as = np.array([-5.0, +5.0])
src_flux = np.array([1.0, 0.7])

# Convert to radians from optical axis
src_theta = np.sqrt(src_x_as ** 2 + src_y_as ** 2) * ARCSEC
src_phi0 = np.arctan2(src_y_as, src_x_as)

# Generate noisy modulation lightcurves for each grid
n_phi = 360
phi_rot = np.linspace(0.0, 2.0 * np.pi, n_phi, endpoint=False)

lc = np.zeros((len(pitches), n_phi))
for i, p in enumerate(pitches):
    for j in range(len(src_flux)):
        lc[i] += src_flux[j] * rmc_modulation(
            phi_rot, src_theta[j], src_phi0[j], pitch=p, L=L
        )
    # add Poisson-like noise (5 % of mean)
    lc[i] += rng.normal(scale=0.05 * lc[i].mean(), size=n_phi)

fig, axes = plt.subplots(3, 3, figsize=(12, 8), sharex=True)
for i, ax in enumerate(axes.flat):
    ax.plot(np.rad2deg(phi_rot), lc[i], lw=0.8)
    ax.set_title(f'RMC {i+1}: p = {pitches[i]*1e6:.0f} um')
    ax.grid(alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel('Rotation phase [deg]')
plt.suptitle('Simulated RHESSI lightcurves — two-footpoint flare')
plt.tight_layout()
plt.show()

In [ ]:
def back_project(lc, pitches, phi_rot, fov_arcsec=60.0, n_pix=121, L=1.55):
    """Simple back-projection image reconstruction from RMC lightcurves.

    Args:
      lc: array (n_grids, n_phi) of modulated count rates.
      pitches: array of grid pitches [m].
      phi_rot: array of rotation phases [rad] for each lc sample.
      fov_arcsec: half-width of square field of view [arcsec].
      n_pix: number of pixels per side.
      L: grid separation [m].

    Returns:
      (image, x_grid_arcsec, y_grid_arcsec) where image has shape (n_pix, n_pix).
    """
    x = np.linspace(-fov_arcsec, fov_arcsec, n_pix)
    y = np.linspace(-fov_arcsec, fov_arcsec, n_pix)
    X, Y = np.meshgrid(x, y)
    theta_grid = np.sqrt(X ** 2 + Y ** 2) * ARCSEC
    phi0_grid = np.arctan2(Y, X)

    image = np.zeros_like(theta_grid)
    # Subtract DC component to keep modulation only
    lc_demean = lc - lc.mean(axis=1, keepdims=True)

    for i, p in enumerate(pitches):
        # For each pixel, compute expected modulation pattern across rotation
        # then dot-product with observed lightcurve.
        for k, ph in enumerate(phi_rot):
            arg = (2.0 * np.pi * L / p) * np.sin(theta_grid) * np.cos(ph - phi0_grid)
            image += lc_demean[i, k] * np.cos(arg)
    return image, x, y


image, x, y = back_project(lc, pitches, phi_rot, fov_arcsec=40.0, n_pix=81)

fig, ax = plt.subplots()
im = ax.imshow(
    image,
    extent=(x[0], x[-1], y[0], y[-1]),
    origin='lower',
    cmap='hot',
)
ax.scatter(src_x_as, src_y_as, s=80, edgecolor='cyan', facecolor='none', lw=2,
           label='True footpoints')
ax.set_xlabel('X [arcsec]')
ax.set_ylabel('Y [arcsec]')
ax.set_title('Back-projection (dirty) image — two-footpoint flare')
ax.legend(loc='lower left')
plt.colorbar(im, ax=ax, label='Brightness (arb)')
plt.tight_layout()
plt.show()

The back-projection image clearly shows two peaks at the input footpoint locations, surrounded by characteristic ring/sidelobe artefacts of the dirty beam. CLEAN, MEM and Pixon iteratively remove these sidelobes; here we keep the *raw* dirty image to visualise the instrument response.

Back-projection 영상은 입력 footpoint 두 위치에 명확한 peak를 보이며, dirty beam의 ring/sidelobe artefact가 둘러싸고 있다. CLEAN, MEM, Pixon이 반복적으로 이 sidelobe를 제거하지만, 여기서는 기기 응답을 시각화하기 위해 *raw* dirty image만 표시한다.

## Part 3: Hard X-ray Bremsstrahlung Spectrum / Hard X-ray 제동복사 스펙트럼

We compute the photon spectrum that RHESSI would observe from a flare with both a hot thermal component and a power-law nonthermal electron distribution.

* Thermal: optically thin emission from a Maxwellian plasma at temperature $T$ with emission measure EM.
* Nonthermal: thick-target bremsstrahlung from a power-law electron flux $F(E) \propto E^{-\delta}$.

We then add the RHESSI front-segment effective area and a Gaussian energy resolution kernel to produce the predicted *count* spectrum.

RHESSI가 flare에서 관측할 광자 스펙트럼을 thermal과 nonthermal 두 성분으로 계산한다.

* Thermal: 온도 $T$, emission measure EM의 Maxwellian 광학적 박막 방출.
* Nonthermal: power-law 전자 flux $F(E)\propto E^{-\delta}$의 thick-target bremsstrahlung.

이어서 front segment 유효 면적과 Gaussian 에너지 분해능 kernel을 합성해 *count* 스펙트럼을 만든다.

In [ ]:
def thermal_photon_spectrum(eps_keV, T_MK=20.0, EM_cm3=1e49):
    """Optically thin thermal bremsstrahlung photon flux at 1 AU.

    Args:
      eps_keV: photon energies [keV].
      T_MK: plasma temperature [MK].
      EM_cm3: emission measure [cm^-3].

    Returns:
      Photon flux [photons s^-1 cm^-2 keV^-1].
    """
    kT = 0.0862 * T_MK  # keV (kB*T with T in MK)
    norm = 8.1e-39  # rough non-relativistic bremsstrahlung coefficient
    return norm * EM_cm3 / np.sqrt(kT) * np.exp(-eps_keV / kT) / eps_keV


def thicktarget_photon_spectrum(eps_keV, A=1.0, delta=4.0, eps_break=None):
    """Power-law thick-target bremsstrahlung photon flux (schematic).

    For F(E) ~ E^{-delta} thick-target the photon spectrum has slope (delta-1).

    Args:
      eps_keV: photon energies [keV].
      A: amplitude at 50 keV.
      delta: electron power-law index.
      eps_break: optional spectral break in keV.

    Returns:
      Photon flux [photons s^-1 cm^-2 keV^-1].
    """
    photon_index = delta - 1.0
    spec = A * (eps_keV / 50.0) ** (-photon_index)
    if eps_break is not None:
        steep = A * (eps_break / 50.0) ** (-photon_index)
        steep_spec = steep * (eps_keV / eps_break) ** (-(photon_index + 1.0))
        spec = np.where(eps_keV < eps_break, spec, steep_spec)
    return spec


def rhessi_effective_area(eps_keV):
    """Schematic RHESSI front-segment effective area (cm^2) vs energy [keV].

    Approximate fit to Fig. 4(c) of Lin et al. 2002 (no attenuators).
    """
    A = np.where(
        eps_keV < 10.0,
        32.0 * (eps_keV / 10.0) ** 1.5,
        np.where(
            eps_keV < 100.0,
            32.0 + (60.0 - 32.0) * (eps_keV - 10.0) / 90.0,
            60.0 * (100.0 / eps_keV) ** 0.7,
        ),
    )
    return A


def gaussian_smear(spec, eps_keV, fwhm_keV=1.0):
    """Convolve a spectrum with a Gaussian of given FWHM (logarithmic-safe).

    Args:
      spec: photon flux on eps_keV grid.
      eps_keV: linear energy grid [keV].
      fwhm_keV: detector FWHM [keV].

    Returns:
      Smeared spectrum on the same grid.
    """
    sigma = fwhm_keV / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    de = eps_keV[1] - eps_keV[0]
    n = max(int(5 * sigma / de), 3)
    kernel_x = np.arange(-n, n + 1) * de
    kernel = np.exp(-0.5 * (kernel_x / sigma) ** 2)
    kernel /= kernel.sum()
    return np.convolve(spec, kernel, mode='same')

In [ ]:
eps = np.linspace(3.0, 300.0, 4000)
I_th = thermal_photon_spectrum(eps, T_MK=20.0, EM_cm3=1e49)
I_nt = thicktarget_photon_spectrum(eps, A=0.05, delta=4.5)
I_total = I_th + I_nt

I_smear = gaussian_smear(I_total, eps, fwhm_keV=1.0)
A_eff = rhessi_effective_area(eps)
count_rate = I_smear * A_eff  # counts s^-1 keV^-1

fig, ax = plt.subplots()
ax.loglog(eps, I_th, '--', label='Thermal (T=20 MK, EM=1e49)')
ax.loglog(eps, I_nt, '--', label='Nonthermal thick-target (delta=4.5)')
ax.loglog(eps, I_total, '-k', lw=1.5, label='Total photon flux')
ax.loglog(eps, count_rate, color='red', lw=1.5, label='Count rate (smeared, A_eff)')
ax.set_xlabel('Photon energy [keV]')
ax.set_ylabel('Flux [photons or counts s^-1 cm^-2 keV^-1]')
ax.set_title('Composite RHESSI hard X-ray spectrum')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

The thermal component dominates below ~25 keV; the nonthermal power-law takes over above. The crossover energy is precisely the *thermal-nonthermal transition* that RHESSI's ~1 keV resolution was designed to resolve. With the front-segment effective area, the count-rate spectrum keeps usable signal up to ~200 keV in this synthetic flare.

Thermal 성분은 ~25 keV 이하에서 지배적이며, 그 위는 nonthermal power-law가 우세하다. 두 성분의 교차 에너지가 바로 RHESSI ~1 keV 분해능이 분해하려 한 *thermal-nonthermal 전이*이다. Front segment 유효 면적 적용 시 본 합성 flare에서 ~200 keV까지 유효 신호가 남는다.

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Hard X-ray imager / Hard X-ray 영상기 | 9 RMC pairs, 2.3 arcsec | STIX (Solar Orbiter), focusing optics + RMC hybrid |
| Spectrometer / 분광계 | 9 segmented Ge detectors, ~1 keV FWHM | STIX CdZnTe; FOXSI focusing optics + Si/CdTe |
| Image reconstruction / 영상 재구성 | Back-projection + CLEAN/MEM/Pixon | Same algorithms, plus visibility-domain MEM, compressed sensing |
| Telemetry / 텔레메트리 | Photon-by-photon, 24-bit events | Standard for modern X-ray missions (NuSTAR, IXPE, STIX) |
| Software / 소프트웨어 | IDL/SSW free | Python/sunpy, OSPEX, IDL/SSW still used |
| Mission model / 임무 모델 | First PI-mode SMEX | Standard for SMEX/MIDEX class missions |

**Key takeaway / 핵심 요점**: RHESSI's design — RMC imager + cryogenic Ge spectrometer + photon-by-photon telemetry — is now the canonical reference for hard X-ray solar instrumentation. The same back-projection and forward-fitting algorithms demonstrated above are the foundation of every subsequent RMC analysis pipeline.

**핵심 요점**: RMC imager + 저온 Ge spectrometer + photon-by-photon 텔레메트리라는 RHESSI의 설계는 오늘날 hard X-ray 태양 관측 기기의 표준 참조가 되었다. 위에서 구현한 back-projection과 forward-fitting 알고리즘은 이후 모든 RMC 분석 파이프라인의 기초이다.